# Low-Data Current-Local Confirmation

Use this notebook in Google Colab for the May 2026 low-data confirmation run.

This is not a V7 run. It trains only:

- `classical_conv`
- `non_trainable_quantum`

It confirms the seed-42 low-data pilot with seeds `43` and `44` across train fractions `0.10`, `0.25`, `0.50`, and `1.00`.

Before running this notebook, commit and push the local low-data implementation changes. Colab clones from GitHub, so it cannot see uncommitted local files.

## 1. Runtime Check

Use a GPU runtime. L4 is enough. A100 is acceptable but not required.

In [1]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


Sun May 17 20:26:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. Clone Or Refresh Repo

If this cell says `scripts/run_low_data_grid.py` is missing later, stop and push the local commit first.

In [3]:
import os

REPO_URL = 'https://github.com/necatiincekara/Quanvolutional-Neural-Network.git'
BRANCH = 'master'
REPO_PATH = '/content/Quanvolutional-Neural-Network'

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}

%cd {REPO_PATH}
!git fetch origin
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git log --oneline -n 5


Cloning into '/content/Quanvolutional-Neural-Network'...
remote: Enumerating objects: 789, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 789 (delta 66), reused 93 (delta 41), pack-reused 659 (from 1)
Receiving objects: 100% (789/789), 1.38 MiB | 29.36 MiB/s, done.
Resolving deltas: 100% (409/409), done.
/content/Quanvolutional-Neural-Network
Already on 'master'
Your branch is up to date with 'origin/master'.
From https://github.com/necatiincekara/Quanvolutional-Neural-Network
 * branch            master     -> FETCH_HEAD
Already up to date.
d05478c (HEAD -> master, origin/master, origin/HEAD) docs: Add May 17 low-data extension section to Colab notebook, including new training and aggregation steps for additional seeds
b026a69 docs: Update SKILL.md files and add literature review and V8 decision documents; refine workflow and guidelines for literature-backed tasks
82e988f docs: Update CLAUDE.md and README.md with cu

## 4. Install Dependencies

In [4]:
%cd /content/Quanvolutional-Neural-Network
!pip install -r requirements.txt


/content/Quanvolutional-Neural-Network
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 148.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.6/924.6 kB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 114.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 145.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 37.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 18.5 MB/s eta 0:00:00


## 5. Verify Code And Dataset

Dataset is expected at:

- `/content/drive/MyDrive/set/train`
- `/content/drive/MyDrive/set/test`

In [5]:
%cd /content/Quanvolutional-Neural-Network
!test -f scripts/run_low_data_grid.py || (echo 'Missing scripts/run_low_data_grid.py. Commit and push local changes, then rerun clone/pull cell.' && false)
!test -f scripts/aggregate_low_data.py || (echo 'Missing scripts/aggregate_low_data.py. Commit and push local changes, then rerun clone/pull cell.' && false)
!test -d /content/drive/MyDrive/set/train || (echo 'Missing Drive train set: /content/drive/MyDrive/set/train' && false)
!test -d /content/drive/MyDrive/set/test || (echo 'Missing Drive test set: /content/drive/MyDrive/set/test' && false)
!find /content/drive/MyDrive/set/train -type f | wc -l
!find /content/drive/MyDrive/set/test -type f | wc -l


/content/Quanvolutional-Neural-Network
3428
466


## 6. Smoke Checks

This should be quick. If anything fails here, do not start the confirmation run.

In [6]:
%cd /content/Quanvolutional-Neural-Network
!python -m py_compile src/benchmark_protocol.py train_ablation_local.py train_thesis_models.py scripts/run_low_data_grid.py scripts/aggregate_low_data.py
!python train_ablation_local.py --model classical_conv --test
!python train_thesis_models.py --model thesis_hqnn2 --test
!python scripts/run_low_data_grid.py --protocol-version low_data_confirm_v1 --models classical_conv non_trainable_quantum --fractions 0.10 --seeds 43 --split-seed 42 --device auto


/content/Quanvolutional-Neural-Network
Forward pass OK: torch.Size([2, 1, 32, 32]) -> torch.Size([2, 44])
Total params: 88045, Trainable: 88045
Forward pass OK: torch.Size([2, 4, 16, 16]) -> torch.Size([2, 44])
Total params: 248428, Trainable: 248428
Low-data grid contains 2 commands.
[01/02] /usr/bin/python3 train_ablation_local.py --model classical_conv --seed 43 --split-seed 42 --protocol-version low_data_confirm_v1 --train-fraction 0.10 --fraction-seed 42 --device auto
[02/02] /usr/bin/python3 train_ablation_local.py --model non_trainable_quantum --seed 43 --split-seed 42 --protocol-version low_data_confirm_v1 --train-fraction 0.10 --fraction-seed 42 --device auto
Dry run only. Add --execute to run the grid.


## 7. Run Confirmation Training

This is the main training cell. Run this cell once. Result JSONs and checkpoints are written directly into Drive through symlinked artifact directories.

In [7]:
%cd /content/Quanvolutional-Neural-Network
!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_20260502; \
  mkdir -p "$RUN_DIR/experiments_low_data" "$RUN_DIR/models_low_data" experiments models; \
  if [ -d experiments/low_data ] && [ ! -L experiments/low_data ]; then rsync -a experiments/low_data/ "$RUN_DIR/experiments_low_data/"; fi; \
  rm -rf experiments/low_data models/low_data; \
  ln -s "$RUN_DIR/experiments_low_data" experiments/low_data; \
  ln -s "$RUN_DIR/models_low_data" models/low_data; \
  ls -ld experiments/low_data models/low_data
!python scripts/run_low_data_grid.py \
  --execute \
  --protocol-version low_data_confirm_v1 \
  --models classical_conv non_trainable_quantum \
  --fractions 0.10 0.25 0.50 1.00 \
  --seeds 43 44 \
  --split-seed 42 \
  --device auto


/content/Quanvolutional-Neural-Network
lrwxrwxrwx 1 root root 83 May  2 16:38 experiments/low_data -> /content/drive/MyDrive/quanv_results/low_data_confirm_20260502/experiments_low_data
lrwxrwxrwx 1 root root 78 May  2 16:38 models/low_data -> /content/drive/MyDrive/quanv_results/low_data_confirm_20260502/models_low_data
Low-data grid contains 16 commands.
[01/16] /usr/bin/python3 train_ablation_local.py --model classical_conv --seed 43 --split-seed 42 --protocol-version low_data_confirm_v1 --train-fraction 0.10 --fraction-seed 42 --device auto
[02/16] /usr/bin/python3 train_ablation_local.py --model classical_conv --seed 44 --split-seed 42 --protocol-version low_data_confirm_v1 --train-fraction 0.10 --fraction-seed 42 --device auto
[03/16] /usr/bin/python3 train_ablation_local.py --model classical_conv --seed 43 --split-seed 42 --protocol-version low_data_confirm_v1 --train-fraction 0.25 --fraction-seed 42 --device auto
[04/16] /usr/bin/python3 train_ablation_local.py --model classica

## 8. Aggregate And Copy Artifacts To Drive

Run this after training completes. If the browser disconnected but the runtime is still alive, reconnect and run this cell. If the runtime was fully reset, first rerun sections 2 through 4, then run this cell.

In [ ]:
%cd /content/Quanvolutional-Neural-Network
!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_20260502; \
  mkdir -p "$RUN_DIR/experiments_low_data" "$RUN_DIR/models_low_data" experiments models; \
  rm -rf experiments/low_data models/low_data; \
  ln -s "$RUN_DIR/experiments_low_data" experiments/low_data; \
  ln -s "$RUN_DIR/models_low_data" models/low_data
!python scripts/aggregate_low_data.py
!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_20260502; \
  mkdir -p "$RUN_DIR/experiments_low_data" "$RUN_DIR/docs"; \
  rsync -a experiments/low_data/ "$RUN_DIR/experiments_low_data/"; \
  cp experiments/low_data_summary.json "$RUN_DIR/"; \
  cp docs/LOW_DATA_SUMMARY.md "$RUN_DIR/docs/"; \
  ls -lah "$RUN_DIR"; \
  ls -lah "$RUN_DIR/experiments_low_data" | tail -40


## 9. Inspect Decision Rows

In [ ]:
import json
from pathlib import Path

data = json.loads(Path('experiments/low_data_summary.json').read_text())
for row in data['comparisons']:
    print(
        row['family'],
        'frac=', row['train_fraction'],
        row['quantum_model'], row['quantum_test_acc'],
        'vs', row['classical_model'], row['classical_test_acc'],
        'gap C-Q=', row['gap_classical_minus_quantum'],
        'signal=', row['colab_signal'],
        row['signal_reason'],
    )


## 10. May 17 Low-Data Extension

Run this section only after sections 1-6 have completed successfully. This is not a V7/V8 run. It extends the current-local low-data confirmation from seeds `42,43,44` to `42,43,44,45,46,47`.

Use a GPU runtime, preferably L4. The cell below creates a new Drive-backed run folder:

`/content/drive/MyDrive/quanv_results/low_data_confirm_v2_20260517`

It copies the tracked seed-42 pilot rows and the previous Drive-backed seed-43/44 rows, then trains only seeds `45,46,47` for:

- `classical_conv`
- `non_trainable_quantum`

across fractions `0.10`, `0.25`, `0.50`, and `1.00`.


In [7]:
%cd /content/Quanvolutional-Neural-Network

!git fetch origin
!git pull --ff-only origin master
!test -f scripts/run_low_data_grid.py || (echo 'Missing scripts/run_low_data_grid.py. Push latest repo changes first.' && false)
!test -f scripts/aggregate_low_data.py || (echo 'Missing scripts/aggregate_low_data.py. Push latest repo changes first.' && false)
!test -d /content/drive/MyDrive/set/train || (echo 'Missing Drive train set: /content/drive/MyDrive/set/train' && false)
!test -d /content/drive/MyDrive/set/test || (echo 'Missing Drive test set: /content/drive/MyDrive/set/test' && false)
!test -d /content/drive/MyDrive/quanv_results/low_data_confirm_20260502/experiments_low_data || (echo 'Missing previous low-data Drive rows. Stop and check Drive mount.' && false)

!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_v2_20260517;   PREV_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_20260502;   mkdir -p "$RUN_DIR/experiments_low_data" "$RUN_DIR/models_low_data" "$RUN_DIR/docs" experiments models;   rm -rf experiments/low_data models/low_data;   git restore experiments/low_data || true;   rsync -a experiments/low_data/*.json "$RUN_DIR/experiments_low_data/";   rsync -a "$PREV_DIR/experiments_low_data/" "$RUN_DIR/experiments_low_data/";   rm -rf experiments/low_data models/low_data;   ln -s "$RUN_DIR/experiments_low_data" experiments/low_data;   ln -s "$RUN_DIR/models_low_data" models/low_data;   echo "Staged rows:";   find experiments/low_data -maxdepth 1 -name '*.json' | wc -l;   ls -ld experiments/low_data models/low_data

!python scripts/run_low_data_grid.py   --execute   --protocol-version low_data_confirm_v2_20260517   --models classical_conv non_trainable_quantum   --fractions 0.10 0.25 0.50 1.00   --seeds 45 46 47   --split-seed 42   --device auto   --continue-on-error


/content/Quanvolutional-Neural-Network
From https://github.com/necatiincekara/Quanvolutional-Neural-Network
 * branch            master     -> FETCH_HEAD
Already up to date.
Staged rows:
0
lrwxrwxrwx 1 root root 86 May 17 20:28 experiments/low_data -> /content/drive/MyDrive/quanv_results/low_data_confirm_v2_20260517/experiments_low_data
lrwxrwxrwx 1 root root 81 May 17 20:28 models/low_data -> /content/drive/MyDrive/quanv_results/low_data_confirm_v2_20260517/models_low_data
Low-data grid contains 24 commands.
[01/24] /usr/bin/python3 train_ablation_local.py --model classical_conv --seed 45 --split-seed 42 --protocol-version low_data_confirm_v2_20260517 --train-fraction 0.10 --fraction-seed 42 --device auto
[02/24] /usr/bin/python3 train_ablation_local.py --model classical_conv --seed 46 --split-seed 42 --protocol-version low_data_confirm_v2_20260517 --train-fraction 0.10 --fraction-seed 42 --device auto
[03/24] /usr/bin/python3 train_ablation_local.py --model classical_conv --seed 47 -

## 11. Aggregate May 17 Extension

Run this after the training cell finishes. It writes an n=6 low-data summary into the Colab repo clone and copies summary artifacts back to Drive.


In [8]:
%cd /content/Quanvolutional-Neural-Network

!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_v2_20260517;   mkdir -p "$RUN_DIR/docs";   rm -rf experiments/low_data models/low_data;   ln -s "$RUN_DIR/experiments_low_data" experiments/low_data;   ln -s "$RUN_DIR/models_low_data" models/low_data

!python scripts/aggregate_low_data.py --allow-partial-overwrite
!python scripts/statistical_evidence.py   --json-out experiments/statistical_evidence_low_data_confirm_v2_20260517.json   --md-out docs/STATISTICAL_EVIDENCE_LOW_DATA_CONFIRM_V2_20260517.md

!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_v2_20260517;   cp experiments/low_data_summary.json "$RUN_DIR/low_data_summary_v2_20260517.json";   cp docs/LOW_DATA_SUMMARY.md "$RUN_DIR/docs/LOW_DATA_SUMMARY_v2_20260517.md";   cp experiments/statistical_evidence_low_data_confirm_v2_20260517.json "$RUN_DIR/";   cp docs/STATISTICAL_EVIDENCE_LOW_DATA_CONFIRM_V2_20260517.md "$RUN_DIR/docs/";   echo "Drive run folder:";   ls -lah "$RUN_DIR";   echo "JSON rows:";   find "$RUN_DIR/experiments_low_data" -maxdepth 1 -name '*.json' | wc -l


/content/Quanvolutional-Neural-Network
Wrote experiments/low_data_summary.json
Wrote docs/LOW_DATA_SUMMARY.md
Wrote experiments/statistical_evidence_low_data_confirm_v2_20260517.json
Wrote docs/STATISTICAL_EVIDENCE_LOW_DATA_CONFIRM_V2_20260517.md
Drive run folder:
total 41K
drwx------ 2 root root 4.0K May 18 06:43 docs
drwx------ 2 root root 4.0K May 18 06:42 experiments_low_data
-rw------- 1 root root  12K May 18 06:43 low_data_summary_v2_20260517.json
drwx------ 2 root root 4.0K May 18 06:42 models_low_data
-rw------- 1 root root  17K May 18 06:43 statistical_evidence_low_data_confirm_v2_20260517.json
JSON rows:
56


## 12. Inspect May 17 Decision Rows

Run this last and paste the output back into Codex. The important rows are the current-local comparison rows and whether the quantum model remains ahead at 10% and 25%.


In [9]:
import json
from pathlib import Path

summary_path = Path('experiments/low_data_summary.json')
data = json.loads(summary_path.read_text())

print('Current-local summary rows')
for row in data['summary']:
    if row.get('family') == 'current-local' and row.get('model') in {'classical_conv', 'non_trainable_quantum'}:
        print(
            row['model'],
            'fraction=', row['train_fraction'],
            'runs=', row['runs'],
            'seeds=', row['seeds'],
            'test=', row['test_acc_mean'], '+/-', row['test_acc_std'],
        )

print('
Current-local comparison rows')
for row in data['comparisons']:
    if row.get('family') == 'current-local':
        print(row)


SyntaxError: unterminated string literal (detected at line 18) (1641624358.py, line 18)